# Phase 3 — Feature-Space Channel Clustering

Reduces 24 sEMG channels → 8 representative channels using **Mutual Information (MI)**.

## Method

1. Load all 44 subjects' preprocessed data `(N, 1, 24, 1500)`
2. For each of the 24 channels: extract 50 libemg features using a **1-channel** libemg call (avoids cross-channel feature contamination; 87 scalars/channel)
3. Mean-pool the 26 sliding-window feature vectors per rep → one `(87,)` vector per rep per channel
4. Compute `mutual_info_classif` between each channel's feature vectors and gesture labels
5. Aggregate MI across feature dimensions → one scalar score per channel
6. Select top 8 channels by MI score
7. Save `feat_representative_channels.npy` for the efficient model notebooks

## Why per-channel libemg calls?

libemg includes cross-channel features (e.g. correlation between channels) when given
multi-channel input. This means a 24-channel call produces 3192 features = 133 scalars/channel,
whereas a single-channel call produces 87 scalars — the 46 extra scalars per channel
in the 24-channel case come from cross-channel features spread across the matrix.
Slicing the 24-channel output into 24 equal blocks would mix within- and cross-channel
features in an undefined way, so we extract each channel in isolation.

## Why MI?

MI directly measures how much a channel's features reduce uncertainty about the gesture
class. It is the dominant channel-selection method in the BCI/EMG literature (filter
method family) and is model-free.

Reference: Jiang et al., *Sensors* 2020 — power-correlation ratio for HD-sEMG.

In [ ]:
print(5)

: 

In [ ]:
import os
import sys
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import libemg
from sklearn.feature_selection import mutual_info_classif

warnings.filterwarnings('ignore')

_SRC = os.path.abspath(os.path.join(os.getcwd(), 'src'))
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)
from emg_loader import load_all_subjects

print('libemg version:', libemg.__version__)

: 

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
DATA_DIR     = '/Volumes/KRIS/data/UG_per_subject'
OUT_DIR      = os.path.join(os.getcwd(), 'clustering & analysis')  # output dir

WINDOW_SIZE  = 250   # samples (~49 ms at 5120 Hz)
WINDOW_SHIFT = 50    # samples (~10 ms)
N_CHANNELS   = 24
N_SELECT     = 8
RANDOM_STATE = 42

RING_NAMES  = ['Elbow (ch 0–7)', 'Middle (ch 8–15)', 'Wrist (ch 16–23)']
RING_COLORS = ['#3498db', '#2ecc71', '#e67e22']

: 

## 1. Confirm feature counts

In [ ]:
fe           = libemg.feature_extractor.FeatureExtractor()
FEATURE_LIST = fe.get_feature_list()

_dummy_1ch  = np.random.randn(2, 1,  250).astype(np.float32)
_dummy_24ch = np.random.randn(2, 24, 250).astype(np.float32)
F_1CH  = fe.extract_features(FEATURE_LIST, _dummy_1ch,  array=True).shape[1]
F_24CH = fe.extract_features(FEATURE_LIST, _dummy_24ch, array=True).shape[1]

print(f'1-channel  output: {F_1CH} scalars/window')
print(f'24-channel output: {F_24CH} scalars/window  ({F_24CH // N_CHANNELS} per channel)')
print(f'Cross-channel features per channel: {F_24CH // N_CHANNELS - F_1CH}')
print(f'\n→ Using per-channel libemg calls with {F_1CH} clean within-channel scalars.')

: 

## 2. Load all subjects and pool

In [ ]:
subjects = load_all_subjects(DATA_DIR)

X_all = np.concatenate([X for _, X, _ in subjects], axis=0)  # (N_total, 1, 24, 1500)
y_all = np.concatenate([y for _, _, y in subjects], axis=0)   # (N_total,)

N_REPS = len(y_all)
N_WINS = len(range(0, X_all.shape[-1] - WINDOW_SIZE + 1, WINDOW_SHIFT))  # 26

print(f'Total reps: {N_REPS}   Windows/rep: {N_WINS}')
print(f'X_all shape: {X_all.shape}')
print(f'Class distribution: {dict(zip(*np.unique(y_all, return_counts=True)))}')

## 3. Extract per-channel features and compute Mutual Information

For each channel we: window → libemg (1-ch) → mean-pool windows → MI with y.

Estimated runtime: ~5–15 min depending on hardware.

In [ ]:
channel_mi     = np.zeros(N_CHANNELS)
channel_feats  = np.zeros((N_CHANNELS, N_REPS, F_1CH), dtype=np.float32)

print(f'Processing {N_CHANNELS} channels × {N_REPS} reps × {N_WINS} windows...')

for ch in range(N_CHANNELS):
    ring_name = ['Elbow', 'Middle', 'Wrist'][ch // 8]

    # Stack all windows for this channel across all reps: (N_REPS * N_WINS, 1, WINDOW_SIZE)
    all_windows = []
    for rep_idx in range(N_REPS):
        sig    = X_all[rep_idx, 0, ch, :]            # (1500,)
        starts = range(0, len(sig) - WINDOW_SIZE + 1, WINDOW_SHIFT)
        wins   = np.stack([sig[s:s + WINDOW_SIZE] for s in starts])  # (N_WINS, WINDOW_SIZE)
        all_windows.append(wins)

    all_wins_ch = np.concatenate(all_windows, axis=0)[:, np.newaxis, :]  # (N_REPS*N_WINS, 1, W)

    # Extract features in one batch call → (N_REPS*N_WINS, F_1CH)
    feats_flat = fe.extract_features(FEATURE_LIST,
                                     all_wins_ch.astype(np.float32),
                                     array=True)
    feats_flat = np.nan_to_num(feats_flat, nan=0.0, posinf=0.0, neginf=0.0)

    # Reshape and mean-pool windows → (N_REPS, F_1CH)
    feats_rep = feats_flat.reshape(N_REPS, N_WINS, F_1CH).mean(axis=1)
    channel_feats[ch] = feats_rep

    # MI with gesture labels
    mi = mutual_info_classif(feats_rep, y_all, random_state=RANDOM_STATE)
    channel_mi[ch] = mi.mean()

    print(f'  ch {ch:2d}  {ring_name:6s}  {ch % 8 * 45:3d}°   MI = {channel_mi[ch]:.4f}')

print(f'\nDone. MI range: [{channel_mi.min():.4f}, {channel_mi.max():.4f}]')

## 4. Select top-8 channels

In [ ]:
ranked   = np.argsort(channel_mi)[::-1]    # descending
selected = np.sort(ranked[:N_SELECT])       # keep sorted for easy indexing
dropped  = np.sort(ranked[N_SELECT:])

print(f'Selected channels (0-indexed): {selected.tolist()}')
print(f'Dropped channels:              {dropped.tolist()}')
print()
print(f'{"Ch":>4}  {"Ring":8}  {"Angle":6}  {"MI":>8}')
print('-' * 36)
for rank, ch in enumerate(ranked[:N_SELECT]):
    ring_label = ['Elbow', 'Middle', 'Wrist'][ch // 8]
    print(f'#{rank+1:2d}  ch{ch:<3d}  {ring_label:8s}  {ch % 8 * 45:3d}°    {channel_mi[ch]:.4f}')

## 5. Visualisations

In [ ]:
# ── Bar chart of MI scores ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 4))

ring_bg_colors = ['#dbeafe', '#dcfce7', '#ffedd5']
for ring_idx, (lo, hi) in enumerate([(-.5, 7.5), (7.5, 15.5), (15.5, 23.5)]):
    ax.axvspan(lo, hi, color=ring_bg_colors[ring_idx], alpha=0.5, zorder=0)

bar_colors = ['#e74c3c' if ch in selected else '#94a3b8' for ch in range(N_CHANNELS)]
ax.bar(range(N_CHANNELS), channel_mi, color=bar_colors, edgecolor='white', linewidth=0.5, zorder=2)

threshold = channel_mi[ranked[N_SELECT - 1]]
ax.axhline(threshold, color='#e74c3c', linestyle='--', linewidth=1, alpha=0.6,
           label=f'Top-{N_SELECT} threshold ({threshold:.4f})')

ax.set_xlabel('Channel index (0-indexed)', fontsize=11)
ax.set_ylabel('Mean Mutual Information', fontsize=11)
ax.set_title(f'Channel Importance by Mutual Information — Top {N_SELECT} selected (red)', fontsize=12)
ax.set_xticks(range(N_CHANNELS))

tick_labels = [f'{["E","M","W"][ch//8]}{ch%8+1}\n({ch})' for ch in range(N_CHANNELS)]
ax.set_xticklabels(tick_labels, fontsize=8)

patches = [mpatches.Patch(color=ring_bg_colors[i], label=RING_NAMES[i]) for i in range(3)]
patches += [mpatches.Patch(color='#e74c3c', label='Selected'),
            mpatches.Patch(color='#94a3b8', label='Dropped')]
ax.legend(handles=patches, loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'mi_channel_scores.png'), dpi=150)
plt.show()
print('Saved: mi_channel_scores.png')

In [ ]:
# ── Forearm ring diagram ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(12, 4.5), subplot_kw=dict(polar=True))
fig.suptitle('Selected Channels on Forearm Electrode Rings\n(red = selected, grey = dropped)',
             fontsize=12, y=1.02)

for ring_idx, ax in enumerate(axes):
    # Electrode 0 at 12 o'clock, going clockwise
    angles = np.linspace(np.pi / 2, np.pi / 2 - 2 * np.pi, 8, endpoint=False)

    for i, angle in enumerate(angles):
        ch_global = ring_idx * 8 + i
        is_sel    = ch_global in selected
        color     = '#e74c3c' if is_sel else '#94a3b8'
        size      = 300 if is_sel else 120

        ax.scatter(angle, 1, s=size, c=color, zorder=4 if is_sel else 3,
                   edgecolors='white', linewidths=1.5)

        ha = 'right' if np.cos(angle) < -0.15 else ('left' if np.cos(angle) > 0.15 else 'center')
        va = 'top'   if np.sin(angle) < -0.15 else ('bottom' if np.sin(angle) > 0.15 else 'center')
        ax.text(angle, 1.45, f'ch{ch_global}\n{channel_mi[ch_global]:.3f}',
                ha=ha, va=va, fontsize=7.5,
                fontweight='bold' if is_sel else 'normal',
                color='#c0392b' if is_sel else '#64748b')

    ax.set_ylim(0, 1.75)
    ax.set_yticks([])
    ax.set_xticks([])
    ax.spines['polar'].set_visible(False)
    ax.set_facecolor('#f8fafc')
    ax.set_title(RING_NAMES[ring_idx], pad=12, fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'ring_diagram.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ring_diagram.png')

In [ ]:
# ── Inter-channel feature correlation heatmap ─────────────────────────────────
# Uses per-channel mean feature vectors — rows are channels, columns are features
ch_signatures = channel_feats.mean(axis=1)   # (24, F_1CH)  mean over all reps
corr = np.corrcoef(ch_signatures)            # (24, 24)

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='Pearson correlation')

for ch in selected:
    for line_fn in [ax.axhline, ax.axvline]:
        for offset in [-0.5, 0.5]:
            line_fn(ch + offset, color='black', linewidth=0.8, alpha=0.4)

tick_labels = [f'{ch}\n{["E","M","W"][ch//8]}{ch%8+1}' for ch in range(N_CHANNELS)]
ax.set_xticks(range(N_CHANNELS));  ax.set_xticklabels(tick_labels, fontsize=7)
ax.set_yticks(range(N_CHANNELS));  ax.set_yticklabels(tick_labels, fontsize=7)
ax.set_title('Inter-Channel Feature Correlation (selected channels bordered)', fontsize=11)

for boundary in [7.5, 15.5]:
    ax.axhline(boundary, color='gray', linewidth=1.5, linestyle='--', alpha=0.5)
    ax.axvline(boundary, color='gray', linewidth=1.5, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'channel_correlation_heatmap.png'), dpi=150)
plt.show()
print('Saved: channel_correlation_heatmap.png')

## 6. Save outputs

In [ ]:
np.save(os.path.join(OUT_DIR, 'feat_representative_channels.npy'), selected)
np.save(os.path.join(OUT_DIR, 'feat_channel_mi_scores.npy'),       channel_mi)

print('Saved: feat_representative_channels.npy')
print('Saved: feat_channel_mi_scores.npy')
print()
print('── Summary ──────────────────────────────────────────────────────────')
print(f'Method  : Mutual Information (sklearn mutual_info_classif, mean over {F_1CH} features)')
print(f'Selected: {N_SELECT} of {N_CHANNELS} channels: {selected.tolist()}')
print()
print('Use in efficient model notebooks:')
print("  channels = np.load('clustering & analysis/feat_representative_channels.npy')")
print("  subjects = [(n, X[:, :, channels, :], y) for n, X, y in load_all_subjects(DATA_DIR)]")

## 7. Per-ring coverage check

In [ ]:
ring_counts = [int((selected < 8).sum()), int(((selected >= 8) & (selected < 16)).sum()),
               int((selected >= 16).sum())]

print('Channels per ring in selection:')
for ring_idx, count in enumerate(ring_counts):
    print(f'  {RING_NAMES[ring_idx]:22s}: {count}/8  {"█" * count}')

if min(ring_counts) == 0:
    print()
    print('WARNING: one ring has zero selected channels.')
    print('Anatomical coverage may be poor — consider a constrained selection')
    print('(e.g. force at least 2 channels per ring, then fill remaining slots by MI).')
else:
    print('\nAll three rings are represented.')